# 03 — Data Preparation

This notebook prepares NASA C-MAPSS FD001 for RUL modeling. In Colab, select **Runtime → Run all**. The notebook uses the same automatic download and archive handling as Data Understanding. If both sources fail, upload the archive or all three FD001 TXT files. In local Jupyter, set `INPUT_PATHS` to a ZIP or the three TXT paths, or place the TXT files in the working directory. Dependencies: pandas and NumPy. [NASA dataset description](https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data).

The official test trajectories and labels are reserved for final evaluation.

In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd

REQUIRED = ("train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt")
INPUT_PATHS = [Path(name) for name in REQUIRED] if all(Path(name).is_file() for name in REQUIRED) else []
SPLIT_SEED = 42
TRAIN_ENGINE_COUNT = 80

DOWNLOAD_URLS = (
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
    "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
)
CACHE_DIR = Path(".cache/cmapss_fd001")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def resolve_files(inputs):
    found = {}
    origins = {}
    budget = [0]
    max_total = 256 * 1024**2
    max_file = 64 * 1024**2
    wanted = {name.lower(): name for name in REQUIRED}
    def visit(label, blob, depth=0):
        if depth > 4:
            raise ValueError("ZIP nesting depth exceeds the limit of 4.")
        if len(blob) > max_file:
            raise ValueError(f"File too large: {label}")
        budget[0] += len(blob)
        if budget[0] > max_total:
            raise ValueError("Total read size exceeds the limit.")
        base = label.rsplit("!", 1)[-1].replace("\\", "/").rsplit("/", 1)[-1].lower()
        if zipfile.is_zipfile(BytesIO(blob)):
            with zipfile.ZipFile(BytesIO(blob)) as z:
                for item in z.infolist():
                    if item.is_dir() or "__MACOSX" in item.filename:
                        continue
                    leaf = item.filename.replace("\\", "/").rsplit("/", 1)[-1].lower()
                    if leaf in wanted or leaf.endswith(".zip"):
                        if item.file_size > max_file:
                            raise ValueError(f"Archive member too large: {item.filename}")
                        visit(f"{label}!{item.filename}", z.read(item), depth+1)
        elif base in wanted:
            name = wanted[base]
            if name in found and found[name] != blob:
                raise ValueError(f"Multiple conflicting copies of {name} were found.")
            found[name] = blob
            origins.setdefault(name, []).append(label)
    for label, blob in inputs.items():
        visit(label, blob)
    missing = set(REQUIRED) - set(found)
    if missing:
        raise FileNotFoundError(f"Missing files: {sorted(missing)}. Upload a ZIP or all three TXT files together.")
    return found, origins

def download_zip(url):
    request = Request(url, headers={"User-Agent": "CMPE412-FD001-DataPreparation/1.0"})
    limit = 64 * 1024**2
    with urlopen(request, timeout=30) as response:
        parts, size = [], 0
        while True:
            chunk = response.read(1024**2)
            if not chunk:
                break
            size += len(chunk)
            if size > limit:
                raise ValueError("Download exceeds 64 MiB limit.")
            parts.append(chunk)
    blob = b"".join(parts)
    if not zipfile.is_zipfile(BytesIO(blob)):
        raise ValueError("Response is not a ZIP archive (possibly an HTML error page).")
    return blob


def acquire_files(input_paths, urls, cache_dir):
    if input_paths:
        fileset, locations = resolve_files({str(p): Path(p).read_bytes() for p in input_paths})
        return fileset, locations, "manual local paths"
    for index, url in enumerate(urls, 1):
        cache = cache_dir / f"source_{index}.zip"
        if cache.exists():
            try:
                fileset, locations = resolve_files({f"{url} [cached]": cache.read_bytes()})
                print("Önbellek kullanıldı:", url)
                return fileset, locations, url
            except (ValueError, OSError, zipfile.BadZipFile) as exc:
                print("Önbellek geçersiz, yeniden indirilecek:", str(exc))
                cache.unlink(missing_ok=True)
        try:
            print("İndiriliyor:", url, flush=True)
            blob = download_zip(url)
            fileset, locations = resolve_files({url: blob})
            temporary = cache.with_suffix(".tmp")
            temporary.write_bytes(blob)
            temporary.replace(cache)
            return fileset, locations, url
        except Exception as exc:
            # Ağ ve arşiv hatalarını göster; kullanıcı iptali yakalanmaz.
            print(f"Kaynak başarısız ({type(exc).__name__}): {str(exc)[:250]}")
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Automatic download failed. Set INPUT_PATHS to a ZIP or all three TXT files, then rerun.") from exc
    print("Otomatik indirme başarısız. NASA ZIP dosyasını veya üç FD001 TXT dosyasını birlikte seç.")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No files selected. Rerun this cell.")
    fileset, locations = resolve_files(uploaded)
    return fileset, locations, "manual Colab upload"

raw_files, _, _ = acquire_files(INPUT_PATHS, DOWNLOAD_URLS, CACHE_DIR)
raw_bytes = raw_files["train_FD001.txt"]

settings = [f"setting_{i}" for i in range(1, 4)]
sensors = [f"sensor_{i}" for i in range(1, 22)]
schema = ["unit", "cycle", *settings, *sensors]

def parse_numeric(blob, width, name):
    try:
        text = blob.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        raise ValueError(f"{name}: expected UTF-8 text.") from exc
    rows = []
    for lineno, line in enumerate(text.splitlines(), 1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) != width:
            raise ValueError(f"{name}, line {lineno}: expected {width} columns, found {len(fields)}.")
        try:
            rows.append([float(x) for x in fields])
        except ValueError as exc:
            raise ValueError(f"{name}, line {lineno}: nonnumeric field.") from exc
    if not rows:
        raise ValueError(f"{name}: file is empty.")
    frame = pd.DataFrame(rows)
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"{name}: missing or infinite numeric values are not allowed.")
    return frame

def parse_sensor_file(blob, name):
    frame = parse_numeric(blob, 26, name)
    frame.columns = schema
    ids = frame[["unit", "cycle"]].to_numpy()
    if not np.isfinite(ids).all() or (ids <= 0).any() or (ids != np.floor(ids)).any():
        raise ValueError(f"{name}: unit/cycle must be positive integers.")
    frame[["unit", "cycle"]] = frame[["unit", "cycle"]].astype("int64")
    return frame

raw = parse_sensor_file(raw_bytes, "train_FD001.txt")
if raw.shape != (20_631, 26):
    raise ValueError(f"Expected FD001 training shape (20631, 26), got {raw.shape}.")

if set(raw["unit"].unique()) != set(range(1, 101)) or raw.duplicated(["unit", "cycle"]).any():
    raise ValueError("Expected 100 engines and unique engine-cycle pairs.")
groups = raw.groupby("unit", sort=True)
if not groups["cycle"].first().eq(1).all() or not groups["cycle"].diff().dropna().eq(1).all():
    raise ValueError("Each engine must start at cycle 1 and advance one cycle at a time.")

print(f"Quality checks passed: {len(raw):,} rows, {raw.shape[1]} columns, 100 engines.")
display(raw.head())

Önbellek kullanıldı: https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip
Quality checks passed: 20,631 rows, 26 columns, 100 engines.


,unit,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393.0,2388.0,100.0,38.90,23.4044


## RUL target and engine split

RUL is the final recorded cycle minus the current cycle for each engine. The target uses the complete trajectory to label past observations. The final cycle is used only in this calculation. The 80/20 split is by engine, so every cycle from the same engine stays together.

In [2]:
data = raw.sort_values(["unit", "cycle"]).reset_index(drop=True)
final_cycles = data.groupby("unit")["cycle"].transform("max")
data["RUL"] = final_cycles - data["cycle"]

last_rows = data.groupby("unit").tail(1)
if not data["RUL"].ge(0).all() or not last_rows["RUL"].eq(0).all():
    raise ValueError("RUL must be nonnegative and zero on each final cycle.")
steps = data.groupby("unit")["RUL"].diff().dropna()
if not steps.eq(-1).all():
    raise ValueError("RUL must decrease by one per observed cycle.")

engine_ids = np.sort(data["unit"].unique())
shuffled_ids = np.random.default_rng(SPLIT_SEED).permutation(engine_ids)
train_ids = sorted(shuffled_ids[:TRAIN_ENGINE_COUNT].tolist())
validation_ids = sorted(shuffled_ids[TRAIN_ENGINE_COUNT:].tolist())

train = data[data["unit"].isin(train_ids)].copy()
validation = data[data["unit"].isin(validation_ids)].copy()
if (len(train_ids), len(validation_ids)) != (80, 20):
    raise ValueError("Expected an 80/20 split by engine.")
if set(train["unit"]) & set(validation["unit"]) or len(train) + len(validation) != len(data):
    raise ValueError("Engine groups overlap or fail to cover the dataset.")

split_summary = pd.DataFrame([
    {"split": "train", "engines": len(train_ids), "rows": len(train)},
    {"split": "validation", "engines": len(validation_ids), "rows": len(validation)},
])
display(split_summary)

,split,engines,rows
0,train,80,16527
1,validation,20,4104


## Feature decisions

Check constant settings and sensors using the training engines only. The low-variability flags use the same exploratory thresholds as Data Understanding and remain diagnostic. The baseline uses current measurements as features. Current cycle and engine ID are kept as tracking columns. Scaling can be fitted in the model pipeline using its training data. No RUL cap or rolling features are applied in this baseline.

In [3]:
candidates = [*settings, *sensors]
values = train[candidates]
minimum = values.min()
maximum = values.max()
scale = pd.concat([minimum.abs(), maximum.abs()], axis=1).max(axis=1)
audit = pd.DataFrame({
    "unique_values": values.nunique(),
    "relative_range": (maximum - minimum).div(scale.replace(0, np.nan)).fillna(0),
    "dominant_share": [values[c].value_counts(normalize=True).iloc[0] for c in candidates],
}, index=candidates)
audit.index.name = "column"
audit["constant"] = audit["unique_values"].eq(1)
audit["low_variability_candidate"] = ~audit["constant"] & (
    audit["relative_range"].le(0.001) | audit["dominant_share"].ge(0.99)
)
audit["decision"] = np.where(audit["constant"], "remove", "retain")
audit["reason"] = np.where(audit["constant"], "Constant across training rows",
                           "Available at prediction time and varies in training")
features = audit.index[~audit["constant"]].tolist()
if not features or set(features) & {"unit", "cycle", "RUL", "final_cycle"}:
    raise ValueError("Invalid model feature list.")

correlation = values[features].corr()
upper = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
pairs = correlation.where(upper).stack().rename("pearson_r").reset_index()
pairs.columns = ["feature_a", "feature_b", "pearson_r"]
strong_pairs = pairs.loc[pairs["pearson_r"].abs().ge(0.9)].copy()

display(audit)
display(strong_pairs)
print("Model input columns:", features)

Model input columns: ['setting_1', 'setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


,unique_values,relative_range,dominant_share,constant,low_variability_candidate,decision,reason
column,,,,,,,
setting_1,155,2.000000,0.020572,False,False,retain,Available at prediction time and varies in tra...
setting_2,13,2.000000,0.102196,False,False,retain,Available at prediction time and varies in tra...
setting_3,1,0.000000,1.000000,True,False,remove,Constant across training rows
sensor_1,1,0.000000,1.000000,True,False,remove,Constant across training rows
sensor_2,309,0.005151,0.009197,False,False,retain,Available at prediction time and varies in tra...
sensor_3,2902,0.028369,0.001513,False,False,retain,Available at prediction time and varies in tra...
sensor_4,3870,0.039057,0.000968,False,False,retain,Available at prediction time and varies in tra...
sensor_5,1,0.000000,1.000000,True,False,remove,Constant across training rows
sensor_6,2,0.000463,0.980698,False,True,retain,Available at prediction time and varies in tra...


,feature_a,feature_b,pearson_r
148,sensor_9,sensor_14,0.963886


## Export and handoff

The two table files keep tracking columns beside the approved features and target. Select model inputs using the ordered list in `feature_spec.json`. For Isolation Forest, select the same approved feature list and omit the supervised target.

In [4]:
def prepare_test_last_cycle(sensor_bytes, rul_bytes):
    test = parse_sensor_file(sensor_bytes, "test_FD001.txt")
    if test.shape != (13_096, 26) or test.duplicated(["unit", "cycle"]).any():
        raise ValueError("Expected 13096 test rows with unique engine-cycle pairs.")
    test_groups = test.groupby("unit", sort=True)
    if not test_groups["cycle"].first().eq(1).all() or not test_groups["cycle"].diff().dropna().eq(1).all():
        raise ValueError("Test engines must start at cycle 1 with consecutive ordered cycles.")
    last_rows = test.sort_values(["unit", "cycle"]).groupby("unit").tail(1).copy()
    labels = parse_numeric(rul_bytes, 1, "RUL_FD001.txt")[0].to_numpy()
    # Line i in the official RUL file corresponds to test unit i (1-based).
    if last_rows["unit"].tolist() != list(range(1, 101)) or len(labels) != 100:
        raise ValueError("Expected test units 1–100 and exactly 100 official RUL labels.")
    if (labels < 0).any() or (labels != np.floor(labels)).any():
        raise ValueError("Official test RUL values must be nonnegative integers.")
    last_rows["RUL"] = labels.astype("int64")
    return last_rows


test_last_cycle = prepare_test_last_cycle(raw_files["test_FD001.txt"], raw_files["RUL_FD001.txt"])

output_dir = Path("data_preparation_outputs")
output_dir.mkdir(exist_ok=True)
tracking = ["unit", "cycle"]
export_columns = [*tracking, *features, "RUL"]

train[export_columns].to_csv(output_dir / "train.csv", index=False)
validation[export_columns].to_csv(output_dir / "validation.csv", index=False)
test_last_cycle[export_columns].to_csv(output_dir / "test_last_cycle.csv", index=False)
pd.DataFrame(
    [(u, "train") for u in train_ids] + [(u, "validation") for u in validation_ids],
    columns=["unit", "split"],
).to_csv(output_dir / "engine_split.csv", index=False)
audit.to_csv(output_dir / "feature_audit.csv")
strong_pairs.to_csv(output_dir / "strong_correlations.csv", index=False)
split_summary.to_csv(output_dir / "split_summary.csv", index=False)

spec = {
    "source_file": "train_FD001.txt",
    "source_sha256": hashlib.sha256(raw_bytes).hexdigest(),
    "source_files_sha256": {name: hashlib.sha256(raw_files[name]).hexdigest() for name in REQUIRED},
    "rul_cap": None,
    "target_policy": "Export uncapped true RUL; compare an optional training cap using original validation RUL",
    "test_target_definition": "Official RUL label aligned with test unit IDs 1 through 100",
    "test_rows": len(test_last_cycle),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "split_seed": SPLIT_SEED,
    "engine_split_ratio": "80/20",
    "feature_columns": features,
    "removed_constant_columns": audit.index[audit["constant"]].tolist(),
    "tracking_columns": tracking,
    "target_column": "RUL",
    "target_definition": "Final recorded cycle minus current cycle, within engine",
    "scaling": "not applied",
    "rolling_features": "not applied",
}
(output_dir / "feature_spec.json").write_text(json.dumps(spec, indent=2) + "\n", encoding="utf-8")

handoff = """# FD001 Data Preparation Handoff

Extract the ZIP. Run the following from the directory containing the exported files.

```python
import json
from pathlib import Path

import pandas as pd

spec = json.loads(Path("feature_spec.json").read_text(encoding="utf-8"))
features = spec["feature_columns"]
target = spec["target_column"]
train = pd.read_csv("train.csv")
validation = pd.read_csv("validation.csv")

X_train, y_train = train[features], train[target]
X_validation, y_validation = validation[features], validation[target]
```

Use the same engine split for model comparison. Use unit as the group key if
running grouped cross-validation within the training set. Fit any scaler in
the model pipeline on training rows only, or on the training fold in each CV split.
For Isolation Forest, use the listed feature columns without the RUL target.
For final evaluation, use the prepared table with one row per official test
engine at its last observed cycle and its official RUL label.

```python
test = pd.read_csv("test_last_cycle.csv")
X_test, y_test = test[features], test[target]
```

Apply the chosen model's fitted preprocessing to `X_test` and score its
predictions against `y_test`. Reserve this table for final evaluation.

For modeling, compare training with the linear RUL target and with the target
capped at 125 cycles using `y_train.clip(upper=125)`. Use the same engine split
and score both against the original validation RUL with MAE and RMSE.

Use the 17 listed features as the baseline. Compare validation MAE after
dropping `sensor_6` (two distinct values, about 98% at the dominant value) and
`setting_1`/`setting_2` (FD001 has a single operating condition). In a separate
experiment, add the current `cycle` as a feature and compare against the same
baseline.
"""
(output_dir / "README.md").write_text(handoff, encoding="utf-8")

archive = Path("data_preparation_results.zip")
members = [
    "train.csv", "validation.csv", "test_last_cycle.csv", "engine_split.csv", "feature_audit.csv",
    "strong_correlations.csv", "split_summary.csv", "feature_spec.json", "README.md",
]
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for name in members:
        bundle.write(output_dir / name, arcname=name)

print(f"Created {archive}. In Colab, download it from the Files panel.")

Created data_preparation_results.zip. In Colab, download it from the Files panel.
